In [1]:
import matplotlib.pyplot as plt
import pandas as pd
from sqlalchemy import create_engine

In [18]:
engine = create_engine(
    "mysql+pymysql://root@localhost:3306/climate_watch"
)

df_city = pd.read_sql("SELECT * FROM GlobalLandTemperaturesByCity", con=engine, parse_dates=["dt"])

df_unique_city_id = df_city[["City", "Country", "Latitude", "Longitude"]].drop_duplicates().reset_index(drop=True).reset_index(names="city_id")

df_city = df_city.merge(df_unique_city_id, how="left")
df_city_sorted = df_city.sort_values(["city_id", "dt"])

In [19]:
# Vind steden met grote fluctuaties in temperatuur
df_city_sorted["next_temp"] = df_city_sorted.groupby(["city_id"])["AverageTemperature"].shift(-1)
df_city_sorted["diff_next_temp"] = df_city_sorted["AverageTemperature"] - df_city_sorted["next_temp"]
df_hottest_monthly_fluctuations = df_city_sorted.sort_values("diff_next_temp", ascending=False)
df_coldest_monthly_fluctuations = df_city_sorted.sort_values("diff_next_temp", ascending=True)
